# Module 4.2 — Decorators Deep Dive

### Python Mastery · Track 4: Functional Programming & Metaprogramming

A decorator is a function that takes another function and returns a modified version of it, adding behaviour without changing the original code. Decorators are everywhere in Python: timing, logging, caching, access control, and registration. This module builds them up from closures to parameterised and stacked forms.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- Apply the decorators to your own functions and observe the added behaviour.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Write a basic decorator and apply it with `@`.
2. Handle any arguments with `*args` and `**kwargs`.
3. Preserve a function's identity with `functools.wraps`.
4. Write parameterised decorators (decorators that take arguments).
5. Stack multiple decorators and reason about their order.

**Prerequisites:** Tracks 1 to 3, and Module 4.1.

---

## Part 1 · What a Decorator Is

A decorator is built on closures. It receives a function, defines a new function (the wrapper) that does something extra and then calls the original, and returns that wrapper. The `@decorator` syntax above a definition is simply shorthand for reassigning the function to the decorated version.

In [ ]:
def announce(func):
    """A decorator that prints a message around the call."""
    def wrapper():
        print("before the call")
        result = func()              # call the original function
        print("after the call")
        return result
    return wrapper

def say_hello():
    print("hello")

# The @ syntax is shorthand for: say_hi = announce(say_hi)
@announce
def say_hi():
    print("hi")

say_hi()
print("---")

# Manual form, to show they are equivalent:
decorated = announce(say_hello)
decorated()

## Part 2 · Handling Arguments with `*args` and `**kwargs`

The wrapper above only works for functions with no arguments. To decorate **any** function, the wrapper must accept arbitrary arguments and pass them through using `*args` and `**kwargs`. It should also return whatever the original function returns.

In [ ]:
def trace(func):
    """Print the call and its result, for a function with any signature."""
    def wrapper(*args, **kwargs):
        print(f"calling {func.__name__} with {args} {kwargs}")
        result = func(*args, **kwargs)        # forward all arguments
        print(f"  returned {result!r}")
        return result                          # forward the return value
    return wrapper

@trace
def add(a, b):
    return a + b

@trace
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}"

add(3, 4)
greet("Asha", greeting="Hi")

## Part 3 · Preserving Identity with `functools.wraps`

A problem with the wrappers above is that the decorated function loses its original name and docstring, it now reports as `wrapper`. This breaks documentation and debugging. Applying `functools.wraps(func)` to the wrapper copies the original function's metadata across, so the decorated function keeps its identity. Always use it.

In [ ]:
import functools

def trace_bad(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

def trace_good(func):
    @functools.wraps(func)               # copy name, docstring, and more from func
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@trace_bad
def task_a():
    """Does task A."""

@trace_good
def task_b():
    """Does task B."""

print("without wraps -> name:", task_a.__name__, "| doc:", task_a.__doc__)
print("with wraps    -> name:", task_b.__name__, "| doc:", task_b.__doc__)

## Part 4 · Parameterised Decorators

Sometimes a decorator needs its own configuration, for example "repeat three times." This requires an extra layer: an outer function takes the parameters and returns a decorator, which then wraps the function. So a parameterised decorator is a function that returns a decorator that returns a wrapper, three levels.

In [ ]:
import functools

def repeat(times):
    """A decorator factory: returns a decorator that runs the function 'times' times."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            results = []
            for _ in range(times):
                results.append(func(*args, **kwargs))
            return results
        return wrapper
    return decorator

@repeat(times=3)                 # repeat(3) returns the decorator, which wraps shout
def shout(word):
    return word.upper() + "!"

print(shout("go"))               # ['GO!', 'GO!', 'GO!']

## Part 5 · Stacking Decorators and a Practical Registry

Several decorators can be applied to one function. They are applied **bottom-up**: the decorator nearest the function wraps first, and the one at the top wraps last and therefore runs first when the function is called. Reading the order carefully avoids surprises.

In [ ]:
import functools

def make_logger(label):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            print(f"[{label}] enter")
            result = func(*args, **kwargs)
            print(f"[{label}] exit")
            return result
        return wrapper
    return decorator

@make_logger("outer")            # applied last -> runs first
@make_logger("inner")            # applied first -> closest to the function
def work():
    print("  doing work")

work()
# Order: outer enter -> inner enter -> work -> inner exit -> outer exit

In [ ]:
# A common real use: a decorator that registers functions in a table.
registry = {}

def register(name):
    """Register the decorated function under 'name', leaving it otherwise unchanged."""
    def decorator(func):
        registry[name] = func
        return func               # return the function as-is; we only record it
    return decorator

@register("greet")
def greet(): return "hello"

@register("farewell")
def farewell(): return "goodbye"

print("registered:", list(registry))
print("call by name:", registry["greet"]())

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — A simple wrapping decorator (Easy)

In [ ]:
def excited(func):
    def wrapper():
        return func() + "!!!"
    return wrapper

@excited
def message():
    return "we did it"

print(message())
# Experiment: change the suffix the wrapper adds.

### Example 2 — Passing arguments through (Easy)

In [ ]:
def double_result(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs) * 2
    return wrapper

@double_result
def total(a, b):
    return a + b

print(total(3, 4))   # (3+4) * 2 = 14

### Example 3 — A timing decorator with wraps (Medium)

In [ ]:
import functools

def timed(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # We avoid the real clock for reproducibility; a real timer would use time.perf_counter.
        print(f"running {func.__name__}...")
        result = func(*args, **kwargs)
        print(f"{func.__name__} finished")
        return result
    return wrapper

@timed
def compute(n):
    """Sum the first n integers."""
    return sum(range(n))

print("result:", compute(100))
print("name preserved:", compute.__name__, "| doc:", compute.__doc__)

### Example 4 — Caching with the built-in lru_cache (Medium)

In [ ]:
import functools

@functools.lru_cache(maxsize=None)     # a ready-made caching decorator from the standard library
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print("fib(30):", fib(30))             # fast despite the naive recursion, thanks to caching
print("cache info:", fib.cache_info())

### Example 5 — A parameterised validation decorator (Difficult)

In [ ]:
import functools

def require_positive(*arg_names):
    """Ensure the named keyword arguments are positive before calling."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(**kwargs):
            for name in arg_names:
                if name in kwargs and kwargs[name] <= 0:
                    raise ValueError(f"{name} must be positive")
            return func(**kwargs)
        return wrapper
    return decorator

@require_positive("width", "height")
def rectangle_area(width, height):
    return width * height

print("valid:", rectangle_area(width=4, height=5))
try:
    rectangle_area(width=-1, height=5)
except ValueError as e:
    print("rejected:", e)

### Example 6 — Stacked decorators with clear ordering (Difficult)

In [ ]:
import functools

def tag(name):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            inner = func(*args, **kwargs)
            return f"<{name}>{inner}</{name}>"
        return wrapper
    return decorator

@tag("p")            # applied last -> outermost in the output
@tag("b")            # applied first -> innermost in the output
def content():
    return "hello"

print(content())     # <p><b>hello</b></p>
# The decorator closest to the function wraps first, ending up innermost.

---

## Exercises

**Exercise 1 (Easy).** Write a decorator `shout` that uppercases whatever string the decorated function returns. Apply it to a function returning `"hello"`.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Write a decorator `add_exclamation` that works on a function with any arguments and appends `"!"` to its (string) result. Test it on a function that takes a name.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Write a decorator `logged` that uses `functools.wraps`, prints the function name before calling, and returns the result unchanged. Confirm the decorated function keeps its original `__name__`.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Write a parameterised decorator `repeat(n)` that calls the decorated function `n` times and returns a list of results. Apply it with `n=4` to a function that returns a fixed greeting.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Write a parameterised decorator `prefix(label)` that prepends `"[label] "` to the string result of the decorated function. Stack two of them (`prefix("A")` over `prefix("B")`) and predict, then verify, the output for a function returning `"hi"`.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
def shout(func):
    def wrapper():
        return func().upper()
    return wrapper

@shout
def greet():
    return "hello"

print(greet())

**Solution 2**

In [ ]:
def add_exclamation(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs) + "!"
    return wrapper

@add_exclamation
def greet(name):
    return f"Hello {name}"

print(greet("Asha"))

**Solution 3**

In [ ]:
import functools

def logged(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"calling {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@logged
def compute():
    """Returns a number."""
    return 42

print(compute())
print("name:", compute.__name__)

**Solution 4**

In [ ]:
import functools

def repeat(n):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            return [func(*args, **kwargs) for _ in range(n)]
        return wrapper
    return decorator

@repeat(4)
def hello():
    return "hi"

print(hello())

**Solution 5**

In [ ]:
import functools

def prefix(label):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            return f"[{label}] " + func(*args, **kwargs)
        return wrapper
    return decorator

@prefix("A")     # outermost
@prefix("B")     # innermost
def message():
    return "hi"

print(message())   # [A] [B] hi

---

## Summary and Key Points

- A decorator takes a function and returns a wrapper that adds behaviour and calls the original; `@decorator` is shorthand for reassigning the function to its decorated version.
- A general wrapper accepts `*args` and `**kwargs`, forwards them to the original, and returns its result, so it works with any signature.
- Apply `functools.wraps(func)` to the wrapper to preserve the original name, docstring, and other metadata; omitting it breaks documentation and debugging.
- A parameterised decorator adds a layer: an outer function takes the arguments and returns the decorator, giving three nested functions.
- Stacked decorators apply bottom-up, so the one nearest the function is innermost and the topmost runs first; a no-op decorator that returns the function unchanged is useful for registration.

### Next module: 4.3 — Functional Toolkit

The next module covers the standard functional tools: `map`, `filter`, and `reduce`, and the rich `functools` and `itertools` modules.